<a href="https://colab.research.google.com/github/ZehanQin/ECON5200-Applied-Data-Analytics-in-Econ/blob/main/Lab%2013/Lab_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

In [26]:
url = "https://raw.githubusercontent.com/ZehanQin/ECON5200-Applied-Data-Analytics-in-Econ/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv"

df = pd.read_csv(url)

naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()

print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        18:55:10   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [27]:
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [28]:
multi_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub + Property_Age', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:55:10   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [29]:
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid


res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid


fwl_model = smf.ols('Price_Residuals ~ Age_Residuals -1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802139


In [30]:
"""
OLS Multivariate Regression with Interactive 3D Hyperplane Visualization
========================================================================
Predicts Sale_Price from Property_Age and Distance_to_Tech_Hub,
then renders the fitted regression plane alongside observed data points
using Plotly's graph_objects API.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ──────────────────────────────────────────────
# 1. SYNTHETIC DATA GENERATION
# ──────────────────────────────────────────────
# Replace this block with your actual dataset (pd.read_csv, etc.)
np.random.seed(42)
n = 200

property_age = np.random.uniform(0, 50, n)          # years
distance_to_tech_hub = np.random.uniform(1, 30, n)   # miles

# Ground-truth relationship (with noise):
#   Sale_Price ≈ 500_000 − 3_500·Age − 8_000·Distance + ε
sale_price = (
    500_000
    - 3_500 * property_age
    - 8_000 * distance_to_tech_hub
    + np.random.normal(0, 25_000, n)
)

df = pd.DataFrame({
    "Property_Age": property_age,
    "Distance_to_Tech_Hub": distance_to_tech_hub,
    "Sale_Price": sale_price,
})

# ──────────────────────────────────────────────
# 2. FIT THE OLS MODEL
# ──────────────────────────────────────────────
# sm.add_constant prepends a column of 1s so the model estimates an intercept (β₀).
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])
y = df["Sale_Price"]

model = sm.OLS(y, X).fit()
print(model.summary())

# ──────────────────────────────────────────────
# 3. EXTRACT FITTED COEFFICIENTS
# ──────────────────────────────────────────────
# model.params is a pd.Series indexed by the column names we fed in:
#   Index 0  →  'const'                 (β₀, the intercept)
#   Index 1  →  'Property_Age'          (β₁)
#   Index 2  →  'Distance_to_Tech_Hub'  (β₂)
# The regression equation is:
#   Ŷ = β₀ + β₁·Property_Age + β₂·Distance_to_Tech_Hub

beta_0 = model.params["const"]                  # intercept
beta_1 = model.params["Property_Age"]           # slope for Property_Age
beta_2 = model.params["Distance_to_Tech_Hub"]   # slope for Distance_to_Tech_Hub

print(f"\n--- Extracted Coefficients ---")
print(f"  β₀ (Intercept)          = {beta_0:>12,.2f}")
print(f"  β₁ (Property_Age)       = {beta_1:>12,.2f}")
print(f"  β₂ (Distance_to_Tech_Hub) = {beta_2:>12,.2f}")
print(f"  R²                      = {model.rsquared:>12.4f}")

# ──────────────────────────────────────────────
# 4. BUILD THE MESHGRID FOR THE REGRESSION SURFACE
# ──────────────────────────────────────────────
# WHY a meshgrid?
#   A 2-predictor regression plane lives in 3D space (X₁, X₂, Ŷ).
#   To render that surface Plotly needs a *grid* of (X₁, X₂) pairs
#   and a corresponding Z-value (predicted Ŷ) at every grid node.
#
# STEP-BY-STEP:
#   a) Create 1-D arrays spanning the observed range of each predictor.
#   b) np.meshgrid expands them into two 2-D matrices (age_grid, dist_grid)
#      where every combination of (age, distance) is represented once.
#   c) Plug those matrices into the regression equation element-wise
#      to get the predicted Sale_Price at each grid node.

resolution = 30  # controls how smooth the surface looks

# (a) Linearly spaced 1-D vectors across each predictor's range
age_range  = np.linspace(df["Property_Age"].min(),
                         df["Property_Age"].max(),
                         resolution)
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(),
                         df["Distance_to_Tech_Hub"].max(),
                         resolution)

# (b) Expand into 2-D grids — each is shape (resolution, resolution)
#     age_grid[i, j]  = age_range[j]   (constant across rows)
#     dist_grid[i, j] = dist_range[i]  (constant across columns)
age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# (c) Vectorized prediction using the extracted coefficients
#     This is simply: Ŷ = β₀ + β₁·X₁ + β₂·X₂  applied element-wise
price_surface = beta_0 + (beta_1 * age_grid) + (beta_2 * dist_grid)

# ──────────────────────────────────────────────
# 5. COMPUTE RESIDUALS FOR COLOUR-CODING POINTS
# ──────────────────────────────────────────────
# Residuals (ε̂ = Y − Ŷ) let us visually distinguish overpriced
# and underpriced properties relative to the model's expectation.
df["Predicted"]  = model.fittedvalues
df["Residual"]   = model.resid
df["Abs_Resid"]  = df["Residual"].abs()

# ──────────────────────────────────────────────
# 6. BUILD THE 3D PLOTLY FIGURE
# ──────────────────────────────────────────────
fig = go.Figure()

# --- 6a. Scatter: observed data points ---
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(
        size=4,
        color=df["Residual"],           # colour encodes residual direction
        colorscale="RdBu_r",            # red = overpriced, blue = underpriced
        cmin=-df["Abs_Resid"].quantile(0.95),
        cmax= df["Abs_Resid"].quantile(0.95),
        colorbar=dict(title="Residual ($)", x=1.02),
        opacity=0.85,
        line=dict(width=0.3, color="rgba(40,40,40,0.4)"),
    ),
    name="Observed Sales",
    hovertemplate=(
        "<b>Age:</b> %{x:.1f} yrs<br>"
        "<b>Distance:</b> %{y:.1f} mi<br>"
        "<b>Sale Price:</b> $%{z:,.0f}<br>"
        "<extra></extra>"
    ),
))

# --- 6b. Surface: the fitted regression hyperplane ---
fig.add_trace(go.Surface(
    x=age_grid,           # X-axis grid values
    y=dist_grid,          # Y-axis grid values
    z=price_surface,      # predicted Ŷ at every (X₁, X₂) node
    colorscale=[
        [0, "rgba(70, 130, 180, 0.45)"],   # steel-blue, semi-transparent
        [1, "rgba(70, 130, 180, 0.45)"],
    ],
    showscale=False,
    name="Regression Plane",
    hovertemplate=(
        "<b>Age:</b> %{x:.1f} yrs<br>"
        "<b>Distance:</b> %{y:.1f} mi<br>"
        "<b>Predicted:</b> $%{z:,.0f}<br>"
        "<extra>OLS Plane</extra>"
    ),
    contours=dict(           # optional grid-lines on the surface
        x=dict(show=True, color="rgba(255,255,255,0.15)", width=1),
        y=dict(show=True, color="rgba(255,255,255,0.15)", width=1),
    ),
))

# --- 6c. Drop-lines from each point to the plane (residual whiskers) ---
# These vertical segments make it easy to see how far each
# observation sits above or below the fitted surface.
for _, row in df.iterrows():
    fig.add_trace(go.Scatter3d(
        x=[row["Property_Age"],          row["Property_Age"]],
        y=[row["Distance_to_Tech_Hub"],  row["Distance_to_Tech_Hub"]],
        z=[row["Sale_Price"],            row["Predicted"]],
        mode="lines",
        line=dict(color="rgba(180,180,180,0.25)", width=1),
        showlegend=False,
        hoverinfo="skip",
    ))

# ──────────────────────────────────────────────
# 7. LAYOUT & CAMERA SETTINGS
# ──────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            "Multivariate OLS  ·  Sale Price ~ Property Age + Distance to Tech Hub"
            f"<br><sup>Ŷ = {beta_0:,.0f}  "
            f"{'−' if beta_1 < 0 else '+'} {abs(beta_1):,.0f}·Age  "
            f"{'−' if beta_2 < 0 else '+'} {abs(beta_2):,.0f}·Distance   "
            f"(R² = {model.rsquared:.3f})</sup>"
        ),
        x=0.5,
    ),
    scene=dict(
        xaxis_title="Property Age (years)",
        yaxis_title="Distance to Tech Hub (miles)",
        zaxis_title="Sale Price ($)",
        camera=dict(eye=dict(x=1.8, y=-1.8, z=1.0)),
        aspectratio=dict(x=1, y=1, z=0.7),
    ),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.6)"),
    margin=dict(l=0, r=0, t=80, b=0),
    template="plotly_dark",
    width=1000,
    height=750,
)

fig.show()

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.923
Model:                            OLS   Adj. R-squared:                  0.922
Method:                 Least Squares   F-statistic:                     1177.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          2.79e-110
Time:                        18:55:10   Log-Likelihood:                -2305.9
No. Observations:                 200   AIC:                             4618.
Df Residuals:                     197   BIC:                             4628.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 5.022e+05 